# สกิล 2: เบิร์นซับวิดีโอที่ฟิลิปส่งมา (thai-subburn) — โน้ตบุ๊กเดียวครบทั้งสายงาน

โน้ตบุ๊กนี้คือ **บริการเบิร์นซับแบบแยกเดี่ยว (standalone)** สำหรับวิดีโอสำเร็จรูปที่ฟิลิปส่งมา (บ่อยครั้งทีละ ~6 คลิป) — **ไม่เกี่ยวกับการสร้างวิดีโอ Motion Kit** และไม่ใช้โน้ตบุ๊กเสียงเลย งานเริ่มจากไฟล์วิดีโอสำเร็จรูปของฟิลิปเสมอ

สายงานในโน้ตบุ๊กนี้ (ทำตามลำดับ ห้ามข้าม):
- **(A) รับงาน** — ลิสต์ + ดาวน์โหลดวิดีโอจากโฟลเดอร์รับงานบน Drive ที่ฟิลิปกำหนด ทำงานบนสำเนาเสมอ ต้นฉบับห้ามถูกแตะต้อง
- **(B) ถอดเสียง** — Groq `whisper-large-v3-turbo` เอาเวลาระดับคำของทุกคลิป
- **(C) ร่างบทที่แก้แล้ว** — สร้างบทจากเสียงจริง **เสียงชนะบทเสมอ** แมปคลิปด้วยเนื้อหาที่พูด ไม่ใช่ลำดับไฟล์
- **⛔ (D) จุดหยุดอนุมัติ** — ส่งร่างบททุกคลิปให้ฟิลิปอนุมัติ + ฟิลิปเลือกสไตล์ (viralpop หรือ paper-chips) ก่อนเบิร์น ห้ามข้ามเด็ดขาด
- **(E) เบิร์น** — ด้วยตัวเบิร์นที่ล็อกไว้ของสไตล์ที่เลือกเท่านั้น
- **(F) QC + ส่งมอบ** — ตรวจเฟรมและวัดความดังซ้ำทีละคลิป ส่งมอบเป็นไฟล์ใหม่ทุกชิ้น

ตัวเบิร์นอยู่ในชุด `motion-kit-portable` โฟลเดอร์ `subtitle/` (ใช้ชุดไฟล์ร่วมกับโน้ตบุ๊กเรนเดอร์ แต่เป็นคนละงานกัน):
- **viralpop** (ตัวหลัก): `subtitle/viralpop/scripts/burn_script_aligned.py` — ซับประโยค + คำเหลืองเน้นคำสำคัญ สไตล์ ViralPop
- **paper-chips** (สไตล์ที่ 2): `subtitle/paper-chips/chip_burn.py` — ชิปกระดาษฉีกทีละคำ

อินพุตพื้นฐาน: วิดีโอต้นฉบับ**ที่ยังไม่มีซับ** + บทที่ผ่านการอนุมัติจากฟิลิปแล้ว (บทเกิดจากขั้น B–D ของโน้ตบุ๊กนี้เอง)


# 🚨 IRON RULE — เช็กลิสต์บังคับก่อนเบิร์นทุกครั้ง

- [ ] ใช้ **เฉพาะตัวเบิร์น 2 ตัวในโน้ตบุ๊กนี้เท่านั้น** — **ห้ามใช้ `ffmpeg -vf subtitles=` โดยตรงเด็ดขาด** (เคยทำแล้วได้ซับผิดฟอนต์ ไม่มีคำเหลือง ไม่มี ViralPop — ถูกตีกลับ)
- [ ] ฟอนต์ต้องเป็น **Prompt-Bold.ttf** เท่านั้น (อยู่ในชุดทั้ง `subtitle/fonts/` และ `subtitle/viralpop/fonts/`)
- [ ] บทที่ใช้ต้องเป็น **บทที่ฟิลิปอนุมัติแล้ว** และต้องเทียบกับเสียงจริงก่อน: **เสียงชนะบทเสมอ** — ถ้าในคลิปพูดต่างจากบท ให้แก้บทตามเสียงจริงแล้วส่งฟิลิปอนุมัติก่อนเบิร์น ห้ามเบิร์นบทที่ยังไม่ผ่านการอนุมัติ
- [ ] ข้อความบนจอทุกคำต้องมาจากบทที่อนุมัติ **ห้ามแต่งคำใหม่ใส่เอง**
- [ ] ไฟล์ต้นฉบับ (วิดีโอไม่มีซับ / Video Master) ต้องแยกจากไฟล์ที่เบิร์นแล้ว — เขียนไฟล์ออกใหม่เสมอ ห้ามเขียนทับต้นฉบับ
- [ ] **Exit code ของ viralpop:** `0` = เบิร์น + QA ผ่าน / `2` = **mismatch gate บล็อก** (บทกับเสียงไม่ตรงกัน) = **หยุดและรายงานฟิลิปทันที ห้ามบังคับผ่าน ห้ามแก้เกณฑ์เพื่อหลบด่าน** / `1` = error จาก STT/การเบิร์น/QA — หยุดและรายงานเช่นกัน

- [ ] **ห้ามเบิร์นก่อนผ่านจุดหยุดอนุมัติ (ขั้น D) เด็ดขาด** — บทที่แก้แล้วต้องให้ฟิลิปอนุมัติ และฟิลิปต้องเลือกสไตล์ (viralpop / paper-chips) สำหรับชุดงานนี้ก่อนเสมอ
- [ ] **ต้นฉบับของฟิลิปห้ามถูกแตะต้อง** — ทำงานบนสำเนาในพื้นที่ทำงานเท่านั้น ผลลัพธ์เป็นไฟล์ใหม่ทุกชิ้น ห้ามเขียนทับ/แก้/ลบไฟล์ต้นฉบับ


## ขั้น (A) — รับงาน: ลิสต์และดาวน์โหลดวิดีโอจากโฟลเดอร์รับงาน

ฟิลิปจะบอกโฟลเดอร์รับงานบน Drive ให้ (เช่น โฟลเดอร์ที่เขาวางวิดีโอ ~6 คลิปไว้) เซลล์ถัดไปจะ:
1. mount Google Drive (หรือใช้ไฟล์ที่อัปโหลดไว้ใน `/content/intake/` แทน ถ้าไม่ mount)
2. ลิสต์วิดีโอทุกไฟล์ในโฟลเดอร์รับงาน แล้ว **คัดลอก** เข้าพื้นที่ทำงาน `/content/subburn_work/` เป็นไฟล์ทำงานชื่อ `01`, `02`, ... (ต้นฉบับอยู่บน Drive เหมือนเดิม ไม่แตะต้อง)
3. บันทึก manifest (รายการไฟล์ + ขนาดต้นฉบับ) ไว้ใช้ตรวจตอน QC ว่าต้นฉบับไม่เปลี่ยน

⚠️ เลข `01`, `02` เป็นแค่ชื่อไฟล์ทำงานตามลำดับที่ลิสต์เจอ **ห้ามใช้เลขนี้จับคู่บทหรือเนื้อหา** — การแมปคลิปว่าเนื้อหาอะไร ตัดสินจากเสียงที่ถอดได้ในขั้น (B) เท่านั้น


In [ ]:
import json
import shutil
from pathlib import Path

# === ตั้งค่าตรงนี้: โฟลเดอร์รับงานบน Drive ที่ฟิลิปกำหนด ===
INTAKE_DIR = Path("/content/drive/MyDrive/REPLACE_WITH_INTAKE_FOLDER")  # แก้เป็นโฟลเดอร์จริงที่ฟิลิปบอก
UPLOAD_DIR = Path("/content/intake")  # ทางเลือก: ถ้าไม่อยาก mount Drive ให้ฟิลิปอัปโหลดวิดีโอไว้ที่นี่แทน
MOUNT_DRIVE = True  # ตั้ง False ถ้าใช้วิธีอัปโหลดไฟล์แทน

WORK = Path("/content/subburn_work")
WORK.mkdir(parents=True, exist_ok=True)
VIDEO_EXTS = {".mp4", ".mov", ".mkv", ".m4v", ".webm"}

src_dir = None
if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("mount Drive ไม่ได้:", type(e).__name__, "- จะลองใช้โฟลเดอร์อัปโหลดแทน")
    if INTAKE_DIR.is_dir():
        src_dir = INTAKE_DIR
if src_dir is None and UPLOAD_DIR.is_dir() and any(p.suffix.lower() in VIDEO_EXTS for p in UPLOAD_DIR.iterdir()):
    src_dir = UPLOAD_DIR
if src_dir is None:
    raise SystemExit("STOP: ไม่พบโฟลเดอร์รับงาน — ตรวจ INTAKE_DIR ให้ตรงกับที่ฟิลิปกำหนด หรือให้ฟิลิปอัปโหลดวิดีโอไว้ที่ /content/intake/ ก่อน")

videos = sorted(p for p in src_dir.iterdir() if p.is_file() and p.suffix.lower() in VIDEO_EXTS)
assert videos, "STOP: ไม่พบไฟล์วิดีโอในโฟลเดอร์รับงาน: " + str(src_dir)
print("พบวิดีโอ", len(videos), "ไฟล์ใน", src_dir)

manifest = []
for i, v in enumerate(videos, start=1):
    n = f"{i:02d}"
    work_copy = WORK / (n + v.suffix.lower())
    shutil.copy2(v, work_copy)
    manifest.append({"n": n, "source_name": v.name, "source_size": v.stat().st_size,
                     "video": str(work_copy)})
    print(n, "<-", v.name, f"({v.stat().st_size} bytes)")
(WORK / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("บันทึก manifest แล้ว:", WORK / "manifest.json")
print("หมายเหตุ: เลขไฟล์เป็นแค่ชื่อทำงาน — การแมปเนื้อหาตัดสินจากเสียงในขั้น (B) เท่านั้น")


## ขั้น (b) — เตรียมชุดและติดตั้ง dependency

ถ้าแตกชุด portable ไว้ที่ `/content/motion-kit-portable` แล้ว (เช่น จากโน้ตบุ๊กเรนเดอร์) จะใช้ซ้ำเลย
ไม่อย่างนั้นให้อัปโหลด/โหลด zip แบบเดียวกับโน้ตบุ๊กเรนเดอร์ก่อน แล้วค่อยรันเซลล์นี้

dependency ของ viralpop อยู่ใน `subtitle/viralpop/requirements.txt`: `uharfbuzz`, `fontTools`, `pythainlp`, `groq`, `python-dotenv`
(paper-chips ใช้ Pillow + pythainlp + ffmpeg ซึ่งชุดนี้ครอบคลุมแล้ว)


In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

KIT = Path("/content/motion-kit-portable")
assert (KIT / "subtitle" / "viralpop" / "scripts" / "burn_script_aligned.py").is_file(), "ยังไม่มีชุด portable ที่ /content/motion-kit-portable — แตก zip ก่อนตามโน้ตบุ๊กเรนเดอร์ขั้น (a)"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(KIT / "subtitle" / "viralpop" / "requirements.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pillow", "numpy"], check=False)
print("ffmpeg:", shutil.which("ffmpeg"))
assert shutil.which("ffmpeg"), "ไม่พบ ffmpeg — หยุดและรายงาน"
for f in [KIT / "subtitle" / "fonts" / "Prompt-Bold.ttf", KIT / "subtitle" / "viralpop" / "fonts" / "Prompt-Bold.ttf"]:
    print(f, "มีจริง:", f.is_file())
    assert f.is_file(), "ขาดฟอนต์ Prompt-Bold.ttf: " + str(f)
print("พร้อมเบิร์น")


## ขั้น (B) — ถอดเสียงทุกคลิปด้วย Groq (เวลาระดับคำ)

- โมเดลล็อก: `whisper-large-v3-turbo` เท่านั้น (ห้ามดาวน์เกรด) ภาษา `th`
- คีย์จาก Colab Secrets ชื่อ `GROQ_API_KEY` — **ถ้าไม่มีคีย์ ให้หยุดและบอกฟิลิปทันที** ห้ามประดิษฐ์ timing เองเด็ดขาด
- ผลของแต่ละคลิปเก็บเป็น `NN_stt.json` รูปแบบ `{"words": [{"word","start","end"}, ...], "text": ..., "segments": [...]}` — ใช้ต่อทั้งขั้นร่างบท (C) และขั้นเบิร์น (E)


In [ ]:
import json
import os
import subprocess
from pathlib import Path

WORK = Path("/content/subburn_work")
manifest = json.loads((WORK / "manifest.json").read_text(encoding="utf-8"))

# GROQ_API_KEY จาก Colab Secrets
try:
    from google.colab import userdata
    key = userdata.get("GROQ_API_KEY")
    if key:
        os.environ["GROQ_API_KEY"] = key
        print("ตั้ง GROQ_API_KEY จาก Colab Secrets แล้ว (ไม่แสดงค่า)")
except Exception as e:
    print("อ่าน Colab Secrets ไม่ได้:", type(e).__name__)
if not os.environ.get("GROQ_API_KEY"):
    raise SystemExit("STOP: ไม่มี GROQ_API_KEY ใน Colab Secrets — ถอดเสียงไม่ได้ ให้หยุดและบอกฟิลิป ห้ามประดิษฐ์ timing เอง")

from groq import Groq
client = Groq(api_key=os.environ["GROQ_API_KEY"])

for entry in manifest:
    n = entry["n"]
    video = Path(entry["video"])
    stt_path = WORK / f"{n}_stt.json"
    if stt_path.is_file():
        print(n, "มี STT อยู่แล้ว ข้าม (ลบไฟล์ก่อนถ้าต้องการถอดใหม่)")
        entry["stt"] = str(stt_path)
        continue
    wav = WORK / f"{n}_audio.wav"
    subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", str(video), "-vn", "-ar", "16000", "-ac", "1", str(wav)], check=True)
    with open(wav, "rb") as fh:
        resp = client.audio.transcriptions.create(file=fh, model="whisper-large-v3-turbo", language="th", temperature=0.0, response_format="verbose_json", timestamp_granularities=["word", "segment"])
    data = resp.model_dump() if hasattr(resp, "model_dump") else dict(resp)
    words = data.get("words") or []
    if not words:
        raise SystemExit(f"STOP: คลิป {n} — Groq ไม่คืน word timestamps ห้ามประดิษฐ์ timing เอง หยุดและรายงานฟิลิป")
    clean = [{"word": str(w["word"]), "start": float(w["start"]), "end": float(w["end"])} for w in words]
    out = {"words": clean, "text": data.get("text", ""), "segments": data.get("segments", [])}
    stt_path.write_text(json.dumps(out, ensure_ascii=False, indent=2), encoding="utf-8")
    entry["stt"] = str(stt_path)
    print(n, "ถอดเสียงเสร็จ:", len(clean), "คำ ->", stt_path.name)

(WORK / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("ถอดเสียงครบทุกคลิป — ไปขั้น (C) ร่างบทที่แก้แล้ว")


## ขั้น (C) — ร่างบทที่แก้แล้ว (เสียงชนะบทเสมอ)

เซลล์ถัดไปสร้าง `NN_script_draft.txt` ให้ทุกคลิปจากผลถอดเสียง (1 วลีต่อ 1 บรรทัด ตาม segment ของ STT) จากนั้น **คุณ (operator) ต้องตรวจร่างบทเทียบกับเสียงจริงทีละคลิป** ก่อนส่งฟิลิป:

- ฟังเสียงจริงเทียบคำต่อคำ แก้คำที่ STT ฟังผิด ใช้เฉพาะคำที่ได้ยินชัด **ห้ามเดา ห้ามแต่งคำเติม** ช่วงท้ายที่ไม่มีเสียงพูดไม่ต้องมีซับ
- ถ้าฟิลิปส่ง **บทอ้างอิง** มาด้วย (วางเป็น `NN_ref.txt` ในพื้นที่ทำงาน): บทอ้างอิงเป็นแค่ตัวเทียบ **เสียงชนะบทเสมอ** — เซลล์จะพิมพ์รายการคำที่ต่างกันระหว่างบทอ้างอิงกับ STT ให้ คุณต้องฟังจุดที่ต่างกันนั้นเอง ตัดสินตามเสียงจริง และจดไว้ชี้ให้ฟิลิปเห็นในขั้น (D)
- จุดที่ฟังไม่ชัดจริง ๆ ให้ใส่ `[CHECK:วินาที]` ไว้ในร่างบท — ฟิลิปเป็นคนชี้ขาด ห้ามเดาแทน
- **แมปคลิปด้วยเนื้อหาที่พูดจริง** ถ้าพบว่าลำดับเนื้อหาไม่ตรงลำดับไฟล์ ให้จดไว้บอกฟิลิปด้วย


In [ ]:
import difflib
import json
from pathlib import Path

WORK = Path("/content/subburn_work")
manifest = json.loads((WORK / "manifest.json").read_text(encoding="utf-8"))

for entry in manifest:
    n = entry["n"]
    stt = json.loads(Path(entry["stt"]).read_text(encoding="utf-8"))
    draft_path = WORK / f"{n}_script_draft.txt"
    if not draft_path.is_file():
        lines = []
        segments = stt.get("segments") or []
        if segments:
            lines = [str(s.get("text", "")).strip() for s in segments if str(s.get("text", "")).strip()]
        elif stt.get("text"):
            lines = [stt["text"].strip()]
        assert lines, f"STOP: คลิป {n} ไม่มีข้อความจาก STT ให้ร่างบท"
        draft_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    entry["script_draft"] = str(draft_path)
    print("=" * 60)
    print(f"คลิป {n} ({entry['source_name']}) — ร่างบท: {draft_path.name}")
    print(draft_path.read_text(encoding="utf-8"))

    ref_path = WORK / f"{n}_ref.txt"
    if ref_path.is_file():
        ref_words = ref_path.read_text(encoding="utf-8").split()
        stt_words = [w["word"] for w in stt["words"]]
        sm = difflib.SequenceMatcher(a=ref_words, b=stt_words, autojunk=False)
        diffs = [(tag, ref_words[i1:i2], stt_words[j1:j2]) for tag, i1, i2, j1, j2 in sm.get_opcodes() if tag != "equal"]
        print(f"บทอ้างอิง vs เสียงจริง: ต่างกัน {len(diffs)} จุด — ต้องฟังจุดเหล่านี้เอง เสียงชนะบทเสมอ:")
        for tag, a, b in diffs[:30]:
            print(" ", tag, "| บทอ้างอิง:", " ".join(a) or "(ไม่มี)", "| เสียง/STT:", " ".join(b) or "(ไม่มี)")
    print("ตรวจร่างบทเทียบเสียงจริงทีละคลิป แก้คำผิด ใส่ [CHECK:วินาที] ตรงที่ฟังไม่ชัด แล้วไปขั้น (D)")

(WORK / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")


# ⛔ ขั้น (D) — STOP: หยุดส่งร่างบทให้ฟิลิปอนุมัติ + เลือกสไตล์ก่อนเบิร์น (ห้ามข้ามเด็ดขาด)

**หยุดที่ขั้นนี้ ห้ามรันเซลล์เบิร์นใด ๆ จนกว่าฟิลิปจะอนุมัติ**

สิ่งที่ต้องส่งให้ฟิลิปตอนนี้:
1. ร่างบทที่แก้แล้วของ **ทุกคลิป** (ไฟล์ `NN_script_draft.txt` ที่คุณตรวจเทียบเสียงจริงและแก้แล้วในขั้น C)
2. รายการจุด `[CHECK:วินาที]` ที่ฟังไม่ชัด ให้ฟิลิปชี้ขาด
3. จุดที่บทอ้างอิง (ถ้ามี) ไม่ตรงกับเสียงจริง — ตามรายการ diff จากขั้น (C)
4. ถามฟิลิปว่าจะใช้สไตล์ไหนสำหรับชุดนี้: **viralpop** หรือ **paper-chips** (ฟิลิปเป็นคนเลือก ห้ามเลือกแทน)

เมื่อฟิลิปอนุมัติแล้ว ให้ทำตามนี้ก่อนรันเซลล์ถัดไป:
- แก้ร่างบทตามคำชี้ขาดของฟิลิป แล้วบันทึกฉบับอนุมัติเป็น `NN_script.txt` ทีละคลิปใน `/content/subburn_work/`
- สร้างไฟล์ `/content/subburn_work/approval.json` เนื้อหา: `{"approved": true, "style": "viralpop"}` (หรือ `"paper-chips"`)
- เซลล์ถัดไปคือด่านตรวจอนุมัติ — ถ้าไม่มีไฟล์นี้หรือยังไม่ approved มันจะหยุดทันที นั่นคือพฤติกรรมที่ถูกต้อง **ห้ามสร้างไฟล์อนุมัติขึ้นมาเองโดยไม่มีคำอนุมัติจริงจากฟิลิป**


In [ ]:
import json
from pathlib import Path

WORK = Path("/content/subburn_work")
approval_path = WORK / "approval.json"
if not approval_path.is_file():
    raise SystemExit("STOP: ยังไม่มี approval.json — ต้องส่งร่างบทให้ฟิลิปอนุมัติและให้ฟิลิปเลือกสไตล์ก่อน (ดูขั้น D ด้านบน) ห้ามเบิร์นก่อนอนุมัติเด็ดขาด")
approval = json.loads(approval_path.read_text(encoding="utf-8"))
if approval.get("approved") is not True:
    raise SystemExit("STOP: approval.json ยังไม่ใช่ approved — รอคำอนุมัติจริงจากฟิลิปก่อน ห้ามแก้ไฟล์นี้ให้ผ่านเอง")
style = approval.get("style")
if style not in ("viralpop", "paper-chips"):
    raise SystemExit("STOP: approval.json ต้องระบุ style เป็น 'viralpop' หรือ 'paper-chips' ตามที่ฟิลิปเลือก")

manifest = json.loads((WORK / "manifest.json").read_text(encoding="utf-8"))
for entry in manifest:
    script = WORK / f"{entry['n']}_script.txt"
    if not script.is_file():
        raise SystemExit(f"STOP: คลิป {entry['n']} ยังไม่มีบทฉบับอนุมัติ {script.name} — บันทึกบทที่ฟิลิปอนุมัติแล้วก่อนเบิร์น")
    entry["script"] = str(script)
(WORK / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("ผ่านด่านอนุมัติ: ฟิลิปอนุมัติบทแล้ว สไตล์ที่เลือก =", style)
print("ไปขั้น (E) เบิร์นได้ — ใช้เฉพาะตัวเบิร์นของสไตล์ที่เลือกเท่านั้น")


## ขั้น (E) — เบิร์นด้วยตัวเบิร์นที่ล็อกไว้ (เลือกตามสไตล์ที่ฟิลิปอนุมัติในขั้น D)

ทำ **เฉพาะสไตล์ที่ฟิลิปเลือก** สำหรับชุดนี้:
- เลือก **viralpop** → ใช้เซลล์ชุด (c) ด้านล่าง (ทีละคลิป) หรือเซลล์รันชุดท้ายบท — ไฟล์ออก `NN_sub.mp4`
- เลือก **paper-chips** → ใช้เซลล์ชุด (d) ด้านล่าง — ไฟล์ออก `NN_chip.mp4`

เซลล์เบิร์นด้านล่างคือกลไกที่ผ่านการทดสอบควันจริงแล้ว ใช้ตามเดิม ตั้งค่าอินพุตให้ชี้ไปที่ไฟล์ในพื้นที่ทำงาน `/content/subburn_work/` (`NN` = เลขคลิป, บท = `NN_script.txt` ฉบับอนุมัติ)


## ขั้น (c) — viralpop (ตัวหลัก)

อินพุต:
- `VIDEO_PATH` = วิดีโอต้นฉบับ**ไม่มีซับ** (`.mp4`)
- `SCRIPT_PATH` = บท `.txt` ที่ฟิลิปอนุมัติแล้ว และเทียบเสียงจริงแล้ว (เสียงชนะบท)
- `WORDS_JSON` = `""` เป็นค่าเริ่มต้น → ตัวเบิร์นจะเรียก Groq STT สด ต้องมี `GROQ_API_KEY` (ตั้งใน Colab Secrets ชื่อ `GROQ_API_KEY`)
  ถ้ามีไฟล์เวลาระดับคำจากชุดคิท/โฟลว์ cues อยู่แล้ว ให้ใส่ path ตรงนี้เพื่อข้าม STT สด
- ใช้ `--profile vo` เสมอ และ `--frames-dir` เพื่อเก็บเฟรม QA ไว้ตรวจ


In [ ]:
import os
from pathlib import Path

KIT = Path("/content/motion-kit-portable")

# === ตั้งค่าอินพุตตรงนี้ ===
VIDEO_PATH = Path("/content/video.mp4")
SCRIPT_PATH = Path("/content/script.txt")
WORDS_JSON = ""
OUT_PATH = Path("/content/video_viralpop.mp4")
FRAMES_DIR = Path("/content/qa_viralpop")

assert VIDEO_PATH.is_file(), "ไม่พบวิดีโอต้นฉบับ: " + str(VIDEO_PATH)
assert SCRIPT_PATH.is_file(), "ไม่พบไฟล์บทที่อนุมัติแล้ว: " + str(SCRIPT_PATH)

# GROQ_API_KEY จาก Colab Secrets (ถ้ามี) -> env ให้ตัวเบิร์นใช้
try:
    from google.colab import userdata
    key = userdata.get("GROQ_API_KEY")
    if key:
        os.environ["GROQ_API_KEY"] = key
        print("ตั้ง GROQ_API_KEY จาก Colab Secrets แล้ว (ไม่แสดงค่า)")
    else:
        print("ไม่มี GROQ_API_KEY ใน Secrets")
except Exception as e:
    print("อ่าน Colab Secrets ไม่ได้:", type(e).__name__)
if not WORDS_JSON and not os.environ.get("GROQ_API_KEY"):
    print("WARNING: ไม่มี --words-json และไม่มี GROQ_API_KEY — viralpop จะเรียก STT สดไม่ได้ ให้หยุดและรายงาน หรือใส่ WORDS_JSON")
print("VIDEO:", VIDEO_PATH)
print("SCRIPT:", SCRIPT_PATH)
print("WORDS_JSON:", WORDS_JSON or "(ไม่มี — ใช้ Groq STT สด)")


In [ ]:
import subprocess
import sys
from pathlib import Path

KIT = Path("/content/motion-kit-portable")
VIDEO_PATH = Path("/content/video.mp4")
SCRIPT_PATH = Path("/content/script.txt")
WORDS_JSON = ""
OUT_PATH = Path("/content/video_viralpop.mp4")
FRAMES_DIR = Path("/content/qa_viralpop")

cmd = [sys.executable, str(KIT / "subtitle" / "viralpop" / "scripts" / "burn_script_aligned.py"),
       "--video", str(VIDEO_PATH), "--script", str(SCRIPT_PATH),
       "--fonts", str(KIT / "subtitle" / "viralpop" / "fonts"),
       "--out", str(OUT_PATH), "--profile", "vo", "--frames-dir", str(FRAMES_DIR)]
if WORDS_JSON:
    cmd += ["--words-json", WORDS_JSON]
print("+", " ".join(cmd))
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout[-5000:])
print(r.stderr[-3000:])
print("exit code:", r.returncode)
if r.returncode == 2:
    raise SystemExit("STOP: mismatch gate บล็อก (exit 2) — บทกับเสียงไม่ตรงกัน รายงานฟิลิป ห้ามบังคับผ่านเด็ดขาด")
if r.returncode != 0:
    raise SystemExit("STOP: viralpop error (exit " + str(r.returncode) + ") — หยุดและรายงานตาม output ด้านบน")
print("เบิร์นสำเร็จ:", OUT_PATH)


In [ ]:
from pathlib import Path
from IPython.display import Image, display

FRAMES_DIR = Path("/content/qa_viralpop")
frames = sorted(FRAMES_DIR.glob("*.png")) + sorted(FRAMES_DIR.glob("*.jpg")) if FRAMES_DIR.is_dir() else []
print("เฟรม QA ทั้งหมด:", len(frames))
for f in frames[:6]:
    print(f.name)
    display(Image(filename=str(f)))
if not frames:
    print("ไม่พบเฟรม QA ใน frames-dir — ตรวจ output ของเซลล์เบิร์นอีกครั้งก่อนสรุปผล")


## ขั้น (d) — paper-chips (สไตล์ที่ 2: ชิปกระดาษฉีก)

อินพุตต่างจาก viralpop:
- `--script` = บท 1 วลีต่อ 1 บรรทัด (ข้อความในชิปเอาจากบทนี้)
- `--stt` = JSON เวลาระดับคำ รูปแบบที่ `chip_burn.py` อ่านจริงจากซอร์ส: **ออบเจ็กต์ที่มีคีย์ `words`** คือ `{"words": [{"word": ..., "start": ..., "end": ...}, ...]}` (เวลาเป็นวินาที เลขทศนิยม)

วิธีหา timings ตามลำดับ:
1. **มี GROQ_API_KEY ใน Secrets** → รันเซลล์ถอดเสียงด้านล่าง (Groq `whisper-large-v3-turbo`, word timestamps) จะได้ไฟล์รูปแบบตรงกับที่ chip_burn ต้องการพอดี เพราะผล Groq verbose_json ก็เป็น `{"words": [{"word","start","end"}, ...]}` เช่นกัน
2. **ไม่มี key** → หยุดตามเซลล์ STOP ด้านล่าง หรือใช้ตัวแปลงจาก `words.json` ของ Motion Kit (เซลล์แปลงถัดไป)

⚠️ ข้อเท็จจริงเรื่องรูปแบบที่ตรวจจากไฟล์จริง: `words.json` ของ Motion Kit เป็น **ลิสต์ระดับบนสุด** `[{"word","start","end"}, ...]` ไม่ใช่ออบเจ็กต์ `{"words": [...]}` — ตัวแปลงจึงต้องห่อลิสต์ให้เป็น `{"words": [...]}` ก่อน chip_burn ถึงจะอ่านได้ (ตัวแปลงรองรับทั้งสองรูปแบบเผื่อไฟล์จากแหล่งอื่น)


In [ ]:
import json
import os
import subprocess
from pathlib import Path

VIDEO_PATH = Path("/content/video.mp4")
STT_JSON = Path("/content/stt_groq.json")
AUDIO_WAV = Path("/content/audio_for_stt.wav")

# GROQ_API_KEY จาก Colab Secrets (ถ้ามี)
try:
    from google.colab import userdata
    key = userdata.get("GROQ_API_KEY")
    if key:
        os.environ["GROQ_API_KEY"] = key
except Exception:
    pass

if not os.environ.get("GROQ_API_KEY"):
    raise SystemExit("STOP: ไม่มี GROQ_API_KEY — ถอดเสียงเอา timings ไม่ได้ ดูเซลล์ STOP / ตัวแปลงจาก Motion Kit words.json ด้านล่าง")

subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", str(VIDEO_PATH), "-vn", "-ar", "16000", "-ac", "1", str(AUDIO_WAV)], check=True)

from groq import Groq
client = Groq(api_key=os.environ["GROQ_API_KEY"])
with open(AUDIO_WAV, "rb") as fh:
    resp = client.audio.transcriptions.create(file=fh, model="whisper-large-v3-turbo", language="th", temperature=0.0, response_format="verbose_json", timestamp_granularities=["word", "segment"])
data = resp.model_dump() if hasattr(resp, "model_dump") else dict(resp)
words = data.get("words") or []
if not words:
    raise SystemExit("STOP: Groq ไม่คืน word timestamps — ห้ามประดิษฐ์ timing เอง หยุดและรายงาน")
clean = [{"word": str(w["word"]), "start": float(w["start"]), "end": float(w["end"])} for w in words]
STT_JSON.write_text(json.dumps({"words": clean}, ensure_ascii=False, indent=2), encoding="utf-8")
print("เขียน", STT_JSON, "จำนวนคำ:", len(clean))


### ⛔ STOP ถ้าไม่มี word timings

paper-chips **ต้องใช้เวลาระดับคำจากเสียงจริงเท่านั้น** — ถ้าไม่มี GROQ_API_KEY และไม่มี `words.json` จาก Motion Kit ที่มาจากเสียงจริงของคลิปเดียวกัน
ให้ **หยุดและรายงานฟิลิป** ว่าขาด timings ห้ามแบ่งเวลาตามสัดส่วนความยาวบท ห้ามเดาเวลาเองเด็ดขาด (timing ที่ประดิษฐ์ขึ้นจะทำให้ชิปไม่ตรงเสียงและงานถูกตีกลับ)

เซลล์ด้านล่างเป็นทางสำรองที่ถูกต้องทางเดียว: แปลง `words.json` ของ Motion Kit (ซึ่งมาจากเสียงจริงอยู่แล้ว) ให้เป็นรูปแบบ `{"words": [...]}` ที่ chip_burn อ่าน
สมมติฐานที่ต้องรู้: เวลาใน words.json ต้องเป็นของคลิปเดียวกันและบทที่ใช้เบิร์นต้องตรงกับเสียง (เสียงชนะบท) — ตัวแปลงตรวจแค่โครงสร้างไฟล์ ไม่ได้ตรวจว่าเนื้อหาตรงกัน


In [ ]:
import json
from pathlib import Path

# === ชี้ไปที่ words.json ของ Motion Kit และไฟล์ผลลัพธ์ ===
KIT_WORDS_JSON = Path("/content/motion-kit-portable/jobs/REPLACE_WITH_JOB_ID/words.json")
STT_OUT = Path("/content/stt_from_kit.json")

raw = json.loads(KIT_WORDS_JSON.read_text(encoding="utf-8"))
if isinstance(raw, list):
    words = raw
elif isinstance(raw, dict) and isinstance(raw.get("words"), list):
    words = raw["words"]
else:
    raise SystemExit("STOP: รูปแบบ words.json ไม่รู้จัก — ต้องเป็นลิสต์ หรือออบเจ็กต์ที่มีคีย์ words")

clean = []
for i, w in enumerate(words):
    assert isinstance(w, dict) and w.get("word") is not None, "คำที่ " + str(i) + " ไม่มีคีย์ word"
    clean.append({"word": str(w["word"]), "start": float(w["start"]), "end": float(w["end"])})
assert clean, "STOP: ไม่พบคำใน words.json"
STT_OUT.write_text(json.dumps({"words": clean}, ensure_ascii=False, indent=2), encoding="utf-8")
print("แปลงสำเร็จ:", STT_OUT, "จำนวนคำ:", len(clean))
print("ใช้ไฟล์นี้เป็น STT_JSON ในเซลล์รัน chip_burn ถัดไป")


In [ ]:
import subprocess
import sys
from pathlib import Path

KIT = Path("/content/motion-kit-portable")

# === ตั้งค่าอินพุตตรงนี้ (บท = 1 วลีต่อ 1 บรรทัด) ===
VIDEO_PATH = Path("/content/video.mp4")
SCRIPT_PATH = Path("/content/script_phrases.txt")
STT_JSON = Path("/content/stt_groq.json")
OUT_PATH = Path("/content/video_paper_chips.mp4")
QC_FRAME = Path("/content/qc_paper_chips.jpg")

assert VIDEO_PATH.is_file() and SCRIPT_PATH.is_file() and STT_JSON.is_file(), "อินพุตไม่ครบ — ตรวจ VIDEO_PATH / SCRIPT_PATH / STT_JSON อีกครั้ง"
cmd = [sys.executable, str(KIT / "subtitle" / "paper-chips" / "chip_burn.py"),
       "--video", str(VIDEO_PATH), "--script", str(SCRIPT_PATH),
       "--stt", str(STT_JSON), "--out", str(OUT_PATH)]
print("+", " ".join(cmd))
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout[-5000:])
print(r.stderr[-3000:])
print("exit code:", r.returncode)
if r.returncode != 0:
    raise SystemExit("STOP: chip_burn ไม่สำเร็จ — หยุดและรายงานตาม output ด้านบน")
print("เบิร์นสำเร็จ:", OUT_PATH)
print("ไฟล์ chips ข้างผลลัพธ์:", str(OUT_PATH) + ".chips.json")

# ดึงเฟรม QC 1 เฟรม (ใช้ -ss หลัง -i เพื่อให้ได้เฟรมตรงเวลาจริง) แล้วแสดง
subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", str(OUT_PATH), "-ss", "2.0", "-frames:v", "1", str(QC_FRAME)], check=True)
print("เฟรม QC:", QC_FRAME)


In [ ]:
from pathlib import Path
from IPython.display import Image, display

QC_FRAME = Path("/content/qc_paper_chips.jpg")
print("เฟรม QC มีจริง:", QC_FRAME.is_file())
if QC_FRAME.is_file():
    display(Image(filename=str(QC_FRAME)))
print("ตรวจด้วยตา: ชิปอยู่แถบล่าง (ประมาณ y=1470 จาก 1920), ฟอนต์ Prompt Bold, ข้อความตรงบทที่อนุมัติ, จังหวะตรงเสียง")


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

# เซลล์รันชุด (ทางเลือกสะดวก): เบิร์นทุกคลิปตามสไตล์ที่อนุมัติใน approval.json
# กลไกเดียวกับเซลล์เบิร์นทีละคลิปด้านบนเป๊ะ ๆ — ถ้าอยากตรวจทีละคลิปให้ใช้เซลล์ด้านบนแทน ไม่ต้องรันเซลล์นี้
KIT = Path("/content/motion-kit-portable")
WORK = Path("/content/subburn_work")
approval = json.loads((WORK / "approval.json").read_text(encoding="utf-8"))
style = approval["style"]
manifest = json.loads((WORK / "manifest.json").read_text(encoding="utf-8"))

try:
    from google.colab import userdata
    key = userdata.get("GROQ_API_KEY")
    if key:
        os.environ["GROQ_API_KEY"] = key
except Exception:
    pass

for entry in manifest:
    n = entry["n"]
    video = Path(entry["video"])
    script = Path(entry["script"])
    if style == "viralpop":
        out = WORK / f"{n}_sub.mp4"
        cmd = [sys.executable, str(KIT / "subtitle" / "viralpop" / "scripts" / "burn_script_aligned.py"),
               "--video", str(video), "--script", str(script),
               "--fonts", str(KIT / "subtitle" / "viralpop" / "fonts"),
               "--out", str(out), "--profile", "vo", "--frames-dir", str(WORK / f"qa_{n}")]
    else:
        out = WORK / f"{n}_chip.mp4"
        cmd = [sys.executable, str(KIT / "subtitle" / "paper-chips" / "chip_burn.py"),
               "--video", str(video), "--script", str(script),
               "--stt", str(Path(entry["stt"])), "--out", str(out)]
    print("+", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-3000:])
    print(r.stderr[-1500:])
    print(f"คลิป {n} exit code:", r.returncode)
    entry["out"] = str(out) if r.returncode == 0 else None
    entry["burn_exit"] = r.returncode
    if r.returncode == 2:
        (WORK / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
        raise SystemExit(f"STOP: คลิป {n} — mismatch gate บล็อก (exit 2) บทกับเสียงไม่ตรงกัน รายงานฟิลิปทันที ห้ามบังคับผ่าน")
    if r.returncode != 0:
        (WORK / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
        raise SystemExit(f"STOP: คลิป {n} — ตัวเบิร์น error (exit {r.returncode}) หยุดและรายงานตาม output ด้านบน")

(WORK / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("เบิร์นครบทุกคลิป — ไปขั้น (F) QC และส่งมอบ")


## ขั้น (e) — ปิดงานและรายงาน

รายงานฟิลิปด้วย: ไฟล์ผลลัพธ์ของแต่ละสไตล์ที่ทำ, เฟรม QA, exit code ของตัวเบิร์นทุกตัว, และคำ/จังหวะจุดไหนที่ยังไม่มั่นใจให้ฟิลิปฟังตรวจเอง

หมายเหตุเรื่องเสียง: ตัวเบิร์นทั้งสองตัว **คงเสียงต้นฉบับไว้ตามเดิม** (viralpop ประกอบเสียงเดิม, paper-chips ใช้ `-c:a copy`) — ค่าความดัง (LUFS) ถูกกำหนดตั้งแต่ขั้นเรนเดอร์/เตรียมวิดีโอต้นฉบับแล้ว ขั้นเบิร์นซับไม่ได้ปรับความดัง ถ้าวัด LUFS แล้วไม่อยู่ในเกณฑ์ ให้กลับไปแก้ที่ไฟล์ต้นฉบับ ไม่ใช่แก้ที่ตัวเบิร์น
การ QC ผ่านเครื่องไม่ได้แปลว่าพร้อมโพสต์ — ฟิลิปดู/ฟังฉบับเต็มและอนุมัติเองเป็นด่านสุดท้ายเสมอ


### ส่งมอบ (F) — สรุปทุกคลิปเป็นตารางเดียว

เซลล์ถัดไปตรวจไฟล์ที่เบิร์นแล้วทีละคลิป: ขนาดภาพ, ความยาวเทียบต้นฉบับ, วัดความดังซ้ำด้วย loudnorm (ตัวเลขที่วัดจริง), แคปเฟรมต้น/กลาง/ท้ายไว้ตรวจด้วยตา และยืนยันว่าขนาดไฟล์ต้นฉบับบน Drive ไม่เปลี่ยน (ต้นฉบับไม่ถูกแตะต้อง)

รายงานฟิลิปด้วยตารางสรุป: คลิป | สไตล์ | exit code ของตัวเบิร์น | ผล QC (ขนาดภาพ/ความยาว/LUFS/TP) | จุด [CHECK] ที่ฟิลิปชี้ขาดแล้ว — พร้อมแนบเฟรม QC ให้ดู ห้ามสรุปว่า "ผ่าน" โดยไม่มีตัวเลขจริงรองรับ


In [ ]:
import json
import re
import subprocess
from pathlib import Path

WORK = Path("/content/subburn_work")
manifest = json.loads((WORK / "manifest.json").read_text(encoding="utf-8"))
approval = json.loads((WORK / "approval.json").read_text(encoding="utf-8"))
FRAMES = WORK / "qc_frames"
FRAMES.mkdir(exist_ok=True)

def probe(path):
    r = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0",
                        "-show_entries", "stream=width,height", "-show_entries", "format=duration",
                        "-of", "json", str(path)], capture_output=True, text=True)
    d = json.loads(r.stdout)
    return d["streams"][0]["width"], d["streams"][0]["height"], float(d["format"]["duration"])

def measure_loudness(path):
    r = subprocess.run(["ffmpeg", "-i", str(path), "-af", "loudnorm=print_format=json", "-f", "null", "-"],
                       capture_output=True, text=True)
    m = re.findall(r"\{[^}]*\"input_i\"[^}]*\}", r.stderr)
    if not m:
        return None, None
    d = json.loads(m[-1])
    return float(d["input_i"]), float(d["input_tp"])

rows = []
for entry in manifest:
    n = entry["n"]
    out = entry.get("out")
    if not out or not Path(out).is_file():
        rows.append((n, "ไม่มีไฟล์ผลลัพธ์ — ดู exit code ขั้นเบิร์น"))
        continue
    w, h, dur = probe(out)
    _, _, src_dur = probe(entry["video"])
    lufs, tp = measure_loudness(out)
    for label, t in [("start", 0.5), ("mid", dur / 2), ("end", max(dur - 0.5, 0.1))]:
        subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", str(out), "-ss", f"{t:.2f}",
                        "-frames:v", "1", str(FRAMES / f"{n}_{label}.jpg")], check=False)
    ok_dur = abs(dur - src_dur) <= 0.05
    rows.append((n, f"{w}x{h} | ยาว {dur:.2f}s (ต้นฉบับ {src_dur:.2f}s, ตรง={ok_dur}) | {lufs} LUFS / TP {tp} dB | exit {entry.get('burn_exit')}"))

print("สไตล์ของชุดนี้:", approval.get("style"))
print(f"{'คลิป':<6} ผล QC")
for n, s in rows:
    print(f"{n:<6} {s}")
print("เฟรม QC อยู่ที่:", FRAMES, "— เปิดตรวจด้วยตา: ฟอนต์ Prompt Bold, ไม่มี tofu, ซับไม่ซ้อน, คำเหลือง/ชิปตรงบทที่อนุมัติ")
print("ต้นฉบับ: ตรวจขนาดไฟล์ต้นฉบับบน Drive เทียบ manifest (source_size) อีกครั้งก่อนรายงาน — ต้องเท่าเดิมทุกไฟล์")
print("การ QC ผ่านเครื่องไม่ได้แปลว่าพร้อมโพสต์ — ฟิลิปดู/ฟังฉบับเต็มและอนุมัติเองเป็นด่านสุดท้ายเสมอ")


In [ ]:
from pathlib import Path
from google.colab import files

for p in [Path("/content/video_viralpop.mp4"), Path("/content/video_paper_chips.mp4")]:
    print(p, "มีจริง:", p.is_file(), ("ขนาด " + str(p.stat().st_size) + " bytes") if p.is_file() else "")
    if p.is_file():
        files.download(str(p))
